# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor

# --- 1. Load + build target (Weeks 1/5) ---
df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df = df[df['search_volume'].notna() & df['cpc'].notna()]
df['trend_pct_flipped'] = -df['trend_pct']
lower = df['trend_pct_flipped'].quantile(0.01)
upper = df['trend_pct_flipped'].quantile(0.99)
df['trend_pct_dampened'] = df['trend_pct_flipped'].clip(lower, upper)
df['refresh_score'] = df['trend_pct_dampened'] * df['search_volume'] * df['cpc']
df = df[df['trend_pct'].notna()]
threshold = df['refresh_score'].quantile(0.90)
df['is_priority'] = (df['refresh_score'] >= threshold).astype(int)

# --- 2. Baseline rule (Week 4, on the FULL population, unchanged) ---
value = df['cpc'] * df['search_volume']
declining_rank = df['trend_pct_dampened'].abs().rank(pct=True)
staleness_rank = df['days_since_last_update'].rank(pct=True)

near_miss_mask = df['avg_position'].between(8, 20)
decline_threshold = df.loc[near_miss_mask, 'trend_pct_dampened'].quantile(0.75)
stale_threshold = 200

declining_only_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] < stale_threshold)
stale_only_mask = near_miss_mask & (df['days_since_last_update'] >= stale_threshold) & (df['trend_pct_dampened'] < decline_threshold)
stale_declining_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] >= stale_threshold)

score_declining = value * declining_rank
score_stale = value * staleness_rank
score_stale_declining = value * (declining_rank + staleness_rank)

conditions = [stale_declining_mask, declining_only_mask, stale_only_mask]
score_choices = [score_stale_declining, score_declining, score_stale]
reason_choices = ['NEAR_MISS_STALE_DECLINING', 'NEAR_MISS_DECLINING', 'NEAR_MISS_STALE']

df['baseline_score'] = np.select(conditions, score_choices, default=0)
df['reason_code'] = np.select(conditions, reason_choices, default=None)

# --- 3. Grouped split (Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# --- 4. Build features + fit RF (Week 5) ---
drop_cols = [
    'client_id', 'content_id', 'is_priority', 'trend_pct', 'trend_pct_flipped',
    'trend_pct_dampened', 'refresh_score', 'cpc', 'search_volume',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction',
    'provider_used', 'model_used', 'age_tier', 'impression_tier', 'position_tier',
    'freshness_tier', 'char_count_tier', 'word_count_tier',
    'baseline_score', 'reason_code',
]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['refresh_score']
X_test = test_df.drop(columns=drop_cols)

for col in ['word_count', 'char_count']:
    X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
    X_test[f'{col}_missing'] = X_test[col].isna().astype(int)
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

for col in ['competition', 'scroll_rate']:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

for col in ['competition_level', 'main_intent']:
    X_train[col] = X_train[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')

X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_encoded, y_train)
test_df = test_df.copy()
test_df['rf_pred'] = rf.predict(X_test_encoded)

In [11]:
test_df[['content_id','baseline_score','reason_code','rf_pred']].head()

,content_id,baseline_score,reason_code,rf_pred
0,content_304f48230142,0.0,NaN,2037.735800
1,content_a1fb4e703a9e,0.0,NaN,4455.044240
5,content_d4084a4bc775,0.0,NaN,9460.640516
6,content_9a34b442b552,0.0,NaN,0.000000
19,content_af865035b328,0.0,NaN,20696.355566


In [12]:
baseline_top50 = set(
    test_df.sort_values('baseline_score', ascending=False).head(50)['content_id']
)
rf_top50 = set(
    test_df.sort_values('rf_pred', ascending=False).head(50)['content_id']
)

both = baseline_top50 & rf_top50
rf_only = rf_top50 - baseline_top50
rule_only = baseline_top50 - rf_top50

print('HIGH_CONFIDENCE (both agree):', len(both))
print('MODEL_ONLY (RF only):', len(rf_only))
print('RULE_ONLY (baseline only):', len(rule_only))

HIGH_CONFIDENCE (both agree): 1
MODEL_ONLY (RF only): 49
RULE_ONLY (baseline only): 49


In [13]:
rf_top50_df = test_df.sort_values('rf_pred', ascending=False).head(50)
rf_top50_df['avg_position'].describe()
print((rf_top50_df['avg_position'].between(8,20)).sum(), 'of 50 are in the near-miss zone')

22 of 50 are in the near-miss zone


In [14]:
near_miss_test = test_df[test_df['avg_position'].between(8, 20)]

baseline_nm_top50 = set(near_miss_test.sort_values('baseline_score', ascending=False).head(50)['content_id'])
rf_nm_top50 = set(near_miss_test.sort_values('rf_pred', ascending=False).head(50)['content_id'])

overlap_nm = baseline_nm_top50 & rf_nm_top50
print(len(overlap_nm), 'of 50 overlap when both are restricted to the near-miss zone')

2 of 50 overlap when both are restricted to the near-miss zone


In [15]:
baseline_nm_top50_df = near_miss_test.sort_values('baseline_score', ascending=False).head(50)
rf_nm_top50_df = near_miss_test.sort_values('rf_pred', ascending=False).head(50)

print('Baseline top 50 — avg cpc:', baseline_nm_top50_df['cpc'].mean(), ' avg search_volume:', baseline_nm_top50_df['search_volume'].mean())
print('RF top 50 — avg cpc:', rf_nm_top50_df['cpc'].mean(), ' avg search_volume:', rf_nm_top50_df['search_volume'].mean())

Baseline top 50 — avg cpc: 3.3267999999999995  avg search_volume: 822.8
RF top 50 — avg cpc: 1.303  avg search_volume: 565.2


We provide two separate ranked queues rather than one merged list, because the baseline rule and RF model optimize for genuinely different things: the baseline is value-aware (weights cpc × search_volume directly) but scoped only to the near-miss zone (position 8-20); RF has no value signal but scores across the full page population. Testing confirmed these produce almost entirely different top picks (2/50 overlap even within the shared near-miss pool), and the baseline's picks average 2.5x higher CPC — proof the disagreement is structural, not noise. An editor should treat 'Rule-Based Queue' as value-prioritized, narrow-scope recommendations, and 'Model-Based Queue' as broader-scope, decline-pattern-driven recommendations, and choose based on which lens fits this week's priorities.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses this, for what: FlyRank content/SEO editors, to decide which pages to prioritize for refresh work this week.

Where it stops being valid:

1. The Model-Based Queue ranks pages by decline and engagement patterns only — it has no awareness of commercial value (cpc/search_volume were excluded as leakage). Editors should not assume a page appearing here is commercially valuable to fix; pair it with the Rule-Based Queue or manual value context before prioritizing.\

2. is_priority is a defined formula (top 10% of trend_pct_dampened × cpc × search_volume), not an observed outcome — no page in this dataset has ever been confirmed to actually need a refresh. A page appearing in either queue means it scored high on that formula, not that human review or business context confirms it as a genuine priority. The formula also can't see content quality — a well-written page and a poorly-written page with identical metrics score identically."

3. This dataset is a single 90-day cross-sectional snapshot — no page's before/after outcome from an actual refresh is recorded anywhere. This queue can flag pages worth reviewing based on observed patterns (declining, stale, near-miss), but it cannot claim that refreshing a listed page will cause traffic or ranking to improve. Any improvement depends on execution quality, competitive dynamics, and search algorithm behavior — none of which this data measures."

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

1. Before acting on a Model-Based Queue page, manually check its cpc and search_volume — the model didn't. A page with near-zero commercial value may not be worth the editor's time even if it ranks high.

2. Before committing to a refresh, the editor should open and read the actual page — the formula can't detect whether the writing is already strong or genuinely weak. A high-scoring page with excellent existing content may need a light touch, not a full rewrite."

3. Before committing to a refresh, the editor should open and read the actual page — the formula can't detect whether the writing is already strong or genuinely weak. A high-scoring page with excellent existing content may need a light touch, not a full rewrite."

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

- **Precision drift:** if a monthly spot-check of actioned pages shows precision@20 falling well below the validated 0.45 grouped-split baseline, retrain.
- **New client onboarding:** the model was validated on 32 clients via grouped split — a new client's pages are out-of-distribution until enough history accumulates; treat their pages with lower confidence until a few months of data exist.
- **Value proxy added:** if `cpc`/`search_volume` (or a safer non-leaky value signal) get added to the feature set, the model must be fully retrained and re-evaluated — this is the single highest-priority planned improvement.
- **Formula changes:** if `refresh_score`/`is_priority` definitions change, both the baseline and RF model need to be rebuilt and re-validated from scratch, since the entire evaluation depends on that definition.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [16]:
import os
os.makedirs('../outputs', exist_ok=True)

rule_based_queue.to_csv('../outputs/rule_based_queue.csv', index=False)
model_based_queue.to_csv('../outputs/model_based_queue.csv', index=False)

print("Exported rule_based_queue.csv:", len(rule_based_queue), "rows")
print("Exported model_based_queue.csv:", len(model_based_queue), "rows")

Exported rule_based_queue.csv: 50 rows
Exported model_based_queue.csv: 50 rows


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.